# **Task-1**

In [2]:
import torch
import torch.nn as nn
import torch. nn.functional as F
import random

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- Load Data ----------
with open("TrainingNames.txt", "r") as f:
    names = [line.strip().lower() for line in f if line.strip()]

chars = sorted(list(set("".join(names))))
chars = ['<PAD>', '<SOS>', '<EOS>'] + chars

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

def encode(name):
    seq = [stoi['<SOS>']]
    for ch in name:
        seq.append(stoi[ch])
    seq.append(stoi['<EOS>'])

    return seq

def decode(indices):
    decoded_chars = []
    for i in indices:
        if i not in (stoi['<SOS>'], stoi['<EOS>'], stoi['<PAD>']):
            decoded_chars.append(itos[i])
    return "".join(decoded_chars)

encoded_names = [encode(name) for name in names]

Using device: cuda


In [5]:
# ==============================================================
# MODEL 1 — Vanilla RNN
# --------------------------------------------------------------
'''
vocab_size-> total number of unique characters (like a, b, c, ...)
embed_size-> size of vector used to represent each character
hidden_size-> size of hidden state (memory of RNN)
'''
class VanillaRNN(nn.Module):

    def __init__(self, vocab_size, embed_size, hidden_size):
      super().__init__()
      self.hidden_size = hidden_size
      self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=stoi['<PAD>']) #ignore padding for embedding

      #weight from input to hidden state
      self.Wxh = nn.Parameter(torch.randn(embed_size, hidden_size) * 0.01)
      #weight from prev hidden state to curr hidden state
      self.Whh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
      self.bh  = nn.Parameter(torch.zeros(hidden_size))

      self.Why = nn.Parameter(torch.randn(hidden_size, vocab_size) * 0.01)
      self.by  = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x):
      batch_size, seq_len = x.shape
      h = torch.zeros(batch_size, self.hidden_size, device=x.device)
      outputs = []

      for t in range(seq_len):
        '''
        xt is current input character (embedded vector)
        h is previous hidden state (ht-1)
        we compute new hidden state ht = f(xt, ht-1)
        '''
        xt = self.embed(x[:, t])
        #ht=tanh(xt.Wxh+ht-1.Whh+bh)
        h = torch.tanh(xt @ self.Wxh + h @ self.Whh + self.bh)
        #yt=ht.Why+bo
        outputs.append((h @ self.Why + self.by).unsqueeze(1))

      return torch.cat(outputs, dim=1)


# ==============================================================
# MODEL 2 — Bidirectional LSTM (BiLSTM)
# --------------------------------------------------------------

class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
      super().__init__()
      self.hidden_size = hidden_size

      # Forget gate parameters
      self.Wxf = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
      self.Whf = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
      self.bf  = nn.Parameter(torch.zeros(hidden_size))

      # Input gate parameters
      self.Wxi = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
      self.Whi = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
      self.bi  = nn.Parameter(torch.zeros(hidden_size))

      # Candidate parameters
      self.Wxg = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
      self.Whg = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
      self.bg  = nn.Parameter(torch.zeros(hidden_size))

      # Output gate parameters
      self.Wxo = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
      self.Who = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
      self.bo  = nn.Parameter(torch.zeros(hidden_size))


    def forward(self, x_t, h_prev, c_prev):
      """
      x_t   : input at time t
      h_prev: h_{t-1} (previous hidden state)
      c_prev: c_{t-1} (previous cell state)
      """
      # forget gate
      # f_t = σ(x_t Wxf + h_{t-1} Whf + bf)
      f_t = torch.sigmoid(x_t @ self.Wxf + h_prev @ self.Whf + self.bf)

      # input gate
      # i_t = σ(x_t Wxi + h_{t-1} Whi + bi)
      i_t = torch.sigmoid(x_t @ self.Wxi + h_prev @ self.Whi + self.bi)

      # candidate
      # ĝ_t = tanh(x_t Wxg + h_{t-1} Whg + bg)
      g_t = torch.tanh(x_t @ self.Wxg + h_prev @ self.Whg + self.bg)

      # cell state update
      c_t = f_t * c_prev + i_t * g_t

      # output gate
      # o_t = σ(x_t Wxo + h_{t-1} Who + bo)
      o_t = torch.sigmoid(x_t @ self.Wxo + h_prev @ self.Who + self.bo)

      # hidden state
      # h_t = o_t ⊙ tanh(c_t)
      h_t = o_t * torch.tanh(c_t)

      return h_t, c_t

class BLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
      super().__init__()
      self.hidden_size = hidden_size
      self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=stoi['<PAD>'])
      self.fwd   = LSTMCell(embed_size, hidden_size)
      self.bwd   = LSTMCell(embed_size, hidden_size)
      self.fc    = nn.Linear(2 * hidden_size, vocab_size)

    def forward(self, x):
      batch, seq_len = x.shape
      emb = self.embed(x)                  # (batch, T, embed)

      # Forward pass: left → right
      hf = torch.zeros(batch, self.hidden_size, device=x.device)
      cf = torch.zeros(batch, self.hidden_size, device=x.device)
      fwd_out = []
      for t in range(seq_len):
        # to each lstm cell-> input is (xt,ht-1,ct-1) and output is (ht,ct)
        hf, cf = self.fwd(emb[:, t], hf, cf)
        fwd_out.append(hf)  #forward memory after seeing x_t

      # Backward pass: right → left (over the same prefix)
      hb = torch.zeros(batch, self.hidden_size, device=x.device)
      cb = torch.zeros(batch, self.hidden_size, device=x.device)
      bwd_out = [None] * seq_len
      for t in reversed(range(seq_len)):
          hb, cb = self.bwd(emb[:, t], hb, cb)
          bwd_out[t] = hb   #backward memory from future

      # Concat fwd + bwd at every position → project to vocab
      outputs = []
      for t in range(seq_len):
          h = torch.cat([fwd_out[t], bwd_out[t]], dim=1)  # (batch, 2*hidden)
          outputs.append(self.fc(h).unsqueeze(1))

      return torch.cat(outputs, dim=1)                     # (batch, T, vocab)


# ==============================================================
# MODEL 3 — RNN with Attention
# --------------------------------------------------------------

class RNN_Attention(nn.Module):
    '''
    Has attention for all the past hidden states
    '''
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=stoi['<PAD>'])
        # weights similar to RNN, Wxh for input to hidden state and Whh for ht-1 to ht
        self.Wxh = nn.Parameter(torch.randn(embed_size, hidden_size) * 0.01)
        self.Whh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.bh  = nn.Parameter(torch.zeros(hidden_size))
        # weights for attention
        self.Wa  = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.fc  = nn.Linear(2 * hidden_size, vocab_size)

    def forward(self, x):
        batch, seq_len = x.shape
        h = torch.zeros(batch, self.hidden_size, device=x.device)
        hidden_states, outputs = [], []

        for t in range(seq_len):
          xt = self.embed(x[:, t]) # curr input
          # ht calculation acc to simple RNN
          h  = torch.tanh(xt @ self.Wxh + h @ self.Whh + self.bh)

          hidden_states.append(h)

          #stacking all the prev hidden states [h0,h1,h2.....ht]
          H       = torch.stack(hidden_states, dim=1)
          query   = (h @ self.Wa).unsqueeze(1) #computing query from ht

          #finding attention scores b/w curr hidden state and all past hidden states
          #attention->look at all past steps and pick the important ones and combines them->context
          scores  = (query * H).sum(dim=2)
          #converting into probab
          weights = torch.softmax(scores, dim=1).unsqueeze(2)
          context = (weights * H).sum(dim=1)
          outputs.append(self.fc(torch.cat([h, context], dim=1)).unsqueeze(1))

        return torch.cat(outputs, dim=1)

# Training and Eval
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train(model, data, epochs=120, lr=0.001, batch_size=64):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

    for epoch in range(epochs):
        random.shuffle(data) # shuffle data every epoch
        total_loss, n_batches = 0.0, 0

        for start in range(0, len(data), batch_size): #go batch by batch
            batch = data[start: start + batch_size]
            if len(batch) < 2:
                continue

            # find max length in batch (for padding)
            max_len = 0
            for s in batch:
                if len(s) > max_len:
                    max_len = len(s)

            xs_list = []
            ys_list = []

            for s in batch:

                # xs = all chars except last
                x_seq = s[:-1]
                # ys = all chars except first
                y_seq = s[1:]
                # add padding
                pad_len = max_len - len(s)

                for _ in range(pad_len):
                    x_seq.append(stoi['<PAD>'])
                    y_seq.append(stoi['<PAD>'])

                xs_list.append(x_seq)
                ys_list.append(y_seq)

            xs = torch.tensor(xs_list, device=device)
            ys = torch.tensor(ys_list, device=device)

            #forward pass
            out  = model(xs)
            #compute loss
            loss = F.cross_entropy(out.reshape(-1, vocab_size), ys.reshape(-1),
                                   ignore_index=stoi['<PAD>'])
            #backward pass
            optimizer.zero_grad()
            loss.backward()

            # gradient clipping (to avoid exploding gradients)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch:3d}: Loss = {total_loss / n_batches:.4f}")


def generate(model, max_len=40, temperature=0.85):
    model.eval()
    seq, generated = [stoi['<SOS>']], []

    with torch.no_grad():
        for _ in range(max_len):
          # gradient clipping (to avoid exploding gradients)
          x   = torch.tensor([seq], device=device)
          out = model(x)

          # take last time step output
          last_out = out[0, -1]

          # apply temperature and softmax
          probs = torch.softmax(last_out / temperature, dim=0)

          # sample next character
          idx = torch.multinomial(probs, 1).item()

          # stop if EOS
          if idx == stoi['<EOS>']:
              break

          generated.append(idx)
          seq.append(idx)

    return decode(generated)


def evaluate(model, train_names, num_samples=200):
    generated = [generate(model) for _ in range(num_samples)]
    valid     = [g for g in generated if g]
    novelty   = sum(1 for g in valid if g not in set(train_names)) / num_samples
    diversity = len(set(valid)) / num_samples
    return novelty, diversity, valid

# **Task-2**

In [6]:
# ==============================================================
# Run
# ==============================================================
EMBED_SIZE  = 64
HIDDEN_SIZE = 256
EPOCHS      = 120
LR          = 0.001
BATCH_SIZE  = 64

models = [
    ("RNN",   VanillaRNN(vocab_size, EMBED_SIZE, HIDDEN_SIZE)),
    ("BLSTM", BLSTM(vocab_size, EMBED_SIZE, HIDDEN_SIZE)),
    ("ATTN",  RNN_Attention(vocab_size, EMBED_SIZE, HIDDEN_SIZE)),
]

summary = []

for model_name, model in models:
    print(f"\n{'='*50}")
    print(f"Training  : {model_name}")
    print(f"Params    : {count_params(model):,}")
    print(f"Hidden    : {HIDDEN_SIZE}  |  Embed: {EMBED_SIZE}  |  LR: {LR}  |  Epochs: {EPOCHS}")

    train(model, encoded_names, EPOCHS, LR, BATCH_SIZE)

    novelty, diversity, samples = evaluate(model, names)

    print(f"\n{model_name} Results:")
    print(f"  Novelty  : {novelty:.3f}")
    print(f"  Diversity: {diversity:.3f}")
    print(f"  Samples  :")
    for s in samples[:10]:
        print(f"    - {s}")

    fname = f"{model_name}_generated.txt"
    with open(fname, "w") as f:
        for s in samples:
            f.write(s + "\n")
    print(f"  Saved → {fname}")
    summary.append((model_name, count_params(model), novelty, diversity))

print(f"\n{'='*60}")
print(f"{'Model':<12} {'Params':>10} {'Novelty':>10} {'Diversity':>12}")
print(f"{'-'*60}")
for name, params, nov, div in summary:
    print(f"{name:<12} {params:>10,} {nov:>10.3f} {div:>12.3f}")
print(f"{'='*60}")


Training  : RNN
Params    : 91,806
Hidden    : 256  |  Embed: 64  |  LR: 0.001  |  Epochs: 120
  Epoch   0: Loss = 3.0983
  Epoch  10: Loss = 1.6865
  Epoch  20: Loss = 1.4440
  Epoch  30: Loss = 1.2912
  Epoch  40: Loss = 1.2240
  Epoch  50: Loss = 1.1619
  Epoch  60: Loss = 1.0947
  Epoch  70: Loss = 1.0627
  Epoch  80: Loss = 1.0336
  Epoch  90: Loss = 1.0013
  Epoch 100: Loss = 0.9862
  Epoch 110: Loss = 0.9715
  Epoch 119: Loss = 0.9592

RNN Results:
  Novelty  : 0.970
  Diversity: 1.000
  Samples  :
    - yuvanej siddiqui
    - nakel sidvin
    - bhavanesh kapoor
    - aadhila iyer
    - manan reddy
    - para mehta
    - sumanth ahura
    - artik kotri
    - viresh munjal
    - shivendra shetty
  Saved → RNN_generated.txt

Training  : BLSTM
Params    : 674,718
Hidden    : 256  |  Embed: 64  |  LR: 0.001  |  Epochs: 120
  Epoch   0: Loss = 3.0117
  Epoch  10: Loss = 0.0244
  Epoch  20: Loss = 0.0050
  Epoch  30: Loss = 0.0023
  Epoch  40: Loss = 0.0017
  Epoch  50: Loss = 0.0013